# Model Training

Grid Search Cross-Validation over logistic regression hyperparameters.

**Inputs:** `data/facial_drowsiness_data.csv` (produced by `1_feature_extraction.ipynb`)  
**Outputs:** printed CV results table

In [ ]:
import os, sys

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(base_dir, "modules"))
print("base_dir:", base_dir)

In [ ]:
from logreg import LogRegModel
from pipe import MyPipeline
import pandas as pd

df_train = pd.read_csv(os.path.join(base_dir, "data/facial_drowsiness_data.csv"), index_col=0)
df_train.tail()

## Grid Search Cross Validation

In [ ]:
from pca import DDDPCA
from time import time
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# pcas = [3, 5, 10, 100, -1]
def train_evaluate(X_train, X_test, y_train, y_test, epoch, learning_rate, C):
    model = LogisticRegression(learning_rate=learning_rate,max_iter=epoch, C=C).fit(X_train, y_train)
    before = time()
    predictions = model.predict(X_test)
    avg_time = (time()-before)/len(predictions)
    
    conf = confusion_matrix(y_test, predictions)

    tp, fp, fn, tn = conf[0][0], conf[0][1], conf[1][0], conf[1][1]
    accuracy = (tp+tn) / (tp+fp+fn+tn)
    precision = (tp) / (tp+fp)
    recall = (tp) / (tp+fn)
    f1 = 2 / ((1/recall)+(1/precision))
    return (accuracy, precision, recall, f1, avg_time)


def kfold_cv_modelling(df, epoch, learning_rate, C):
    results = [[] for _ in range(5)]
    k = 10
    n = df.shape[0]
    for i in range(k):
        X_train = data.copy()
        X_test = X_train[i*(n//k):i*(n//k)+n//k]
        X_train = X_train.drop(X_train.index[i*(n//k): i*(n//k)+n//k])
        y_train = X_train.pop("class")
        y_test = X_test.pop("class")
        result = train_evaluate(X_train, X_test, y_train, y_test, epoch, learning_rate, C)
        for i, res in enumerate(result):
            results[i].append(res)
    final_results = []
    for arr in results:
        final_results.append(sum(arr)/len(arr))
    return final_results

def get_results(pcas = [10]):
    epochs = [100, 1000, 2000, 3000]
    learning_rates = [0.001, 0.01, 0.1, 0.5]
    cs = [0.1, 0.5, 1, 5, 10]
    results = []
    for n_pca in pcas:
        data = df.copy()
        if n_pca != -1:
            data = DDDPCA.transform_pca(data, n_pca)

        for epoch in epochs:
            for learning_rate in learning_rates:
                for C in cs:
                    print(f"{epoch} || {learning_rate} || {C}")
                    model_results = kfold_cv_modelling(data, epoch, learning_rate, C)
                    acc, prec, recall, f1, pred_time = model_results
                    results.append({
                        "pca": n_pca,
                        "epoch": epoch,
                        "learning_rate": learning_rate,
                        "C" : C,
                        "accuracy": acc,
                        "precision": prec,
                        "recall": recall,
                        "f1-score": f1,
                        "prediction_time": pred_time
                    })
    return results

In [ ]:
res10 = get_results([10])